# Deploy Cross-Industry Pipeline to Fabric
### Programmatically create the Data Pipeline in your Fabric workspace

This notebook reads `CrossIndustry_Pipeline.json` and creates the Data Pipeline artifact in your current Fabric workspace using the REST API.

**What it does:**
1. Find an existing Data Pipeline with at least one notebook activity
2. Extract its definition via REST API to see how Fabric formats notebook references
3. Apply that discovered format to our CrossIndustry_Pipeline.json
4. Create the new pipeline using the REST API

> **Prerequisites:**
> - Run in Fabric workspace (not local)
> - All notebooks (00-07, ZT_Security_Utils, Pipeline_Logger) imported
> - **At least ONE existing Data Pipeline** with a notebook activity (for reference format discovery)
>
> **If you don't have an existing pipeline:**
> 1. In Fabric UI: New → Data Pipeline → name it 'Test_Pipeline'
> 2. Add ONE Notebook activity (any notebook)
> 3. Save the pipeline
> 4. Run this notebook
> 5. We'll discover the format and you can delete Test_Pipeline afterward

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 1: VALIDATE FABRIC RUNTIME
# ════════════════════════════════════════════════════════════════════════

import json
import requests
import time
import base64

try:
    import sempy.fabric as fabric
    workspace_id = fabric.get_workspace_id()
    access_token = notebookutils.credentials.getToken('pbi')
    print(f"✓ Fabric runtime detected")
    print(f"  Workspace ID: {workspace_id}")
except Exception as e:
    raise RuntimeError(f"Must run in Fabric workspace. Error: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2: DISCOVER NOTEBOOK REFERENCE FORMAT FROM EXISTING PIPELINE
# ════════════════════════════════════════════════════════════════════════
# Get an existing pipeline with a notebook activity to extract the correct
# reference format used by Fabric

print("Searching for 'Test_Pipeline'...\n")

items_df = fabric.list_items()
pipelines = items_df[items_df["Type"] == "DataPipeline"]

if pipelines.empty:
    print("✗ No existing pipelines found")
    print("\n⚠️  SETUP REQUIRED: Create a test pipeline manually:")
    print("   1. In Fabric UI: New → Data Pipeline → name it 'Test_Pipeline'")
    print("   2. Add ONE Notebook activity (any notebook from your workspace)")
    print("   3. Save the pipeline")
    print("   4. Re-run this notebook")
    print("\n   We'll use it to discover the correct reference format, then you can delete it.")
    raise RuntimeError("Need at least one pipeline with a notebook activity for discovery")

# Look specifically for Test_Pipeline
test_pipeline = pipelines[pipelines["Display Name"] == "Test_Pipeline"]

if test_pipeline.empty:
    print("✗ 'Test_Pipeline' not found")
    print(f"\nFound {len(pipelines)} other pipeline(s):")
    for _, p in pipelines.iterrows():
        print(f"  • {p['Display Name']}")
    print("\n⚠️  Please create 'Test_Pipeline' with a notebook activity:")
    print("   1. In Fabric UI: New → Data Pipeline → name it 'Test_Pipeline'")
    print("   2. Add ONE Notebook activity (any notebook)")
    print("   3. Save and re-run this notebook")
    raise RuntimeError("Test_Pipeline not found")

# Use Test_Pipeline to discover format
sample_pipeline = test_pipeline.iloc[0]
sample_id = str(sample_pipeline['Id'])
sample_name = sample_pipeline['Display Name']

print(f"✓ Found '{sample_name}'")
print(f"  ID: {sample_id[:8]}...")
print(f"\nExtracting notebook reference format...")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 3: EXTRACT NOTEBOOK REFERENCE FORMAT FROM EXISTING PIPELINE
# ════════════════════════════════════════════════════════════════════════

url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{sample_id}/getDefinition"
headers = {"Authorization": f"Bearer {access_token}"}

resp = requests.post(url, headers=headers, timeout=30)

if resp.status_code != 200:
    raise RuntimeError(f"Failed to get pipeline definition: {resp.status_code} - {resp.text[:500]}")

definition = resp.json()
parts = definition.get('definition', {}).get('parts', [])

# Find the pipeline-content.json part
content_part = next((p for p in parts if 'pipeline-content' in p.get('path', '')), None)

if not content_part:
    raise RuntimeError(f"Pipeline '{sample_name}' has no pipeline-content.json")

# Decode the base64 payload
pipeline_json = json.loads(base64.b64decode(content_part['payload']))

# Look for notebook activities
activities = pipeline_json.get('properties', {}).get('activities', [])

print(f"\nFound {len(activities)} activity(ies) in pipeline:")
for act in activities:
    act_type = act.get('type', 'Unknown')
    act_name = act.get('name', 'Unnamed')
    print(f"  • {act_name} (type: {act_type})")

# Look for notebook activities (Fabric uses 'TridentNotebook' type)
notebook_activities = [a for a in activities if a.get('type') in ['TridentNotebook', 'SynapseNotebook', 'Notebook', 'ExecuteNotebook']]

if not notebook_activities:
    print(f"\n✗ No notebook activities found")
    print(f"\n⚠️  Please check that Test_Pipeline has a Notebook activity")
    raise RuntimeError("Sample pipeline needs at least one notebook activity")

# Extract the reference format
sample_activity = notebook_activities[0]

# Debug: Show the full structure
print(f"\n🔍 Debug - Activity structure:")
print(f"  Type: {sample_activity.get('type')}")
print(f"  Name: {sample_activity.get('name')}")
print(f"  typeProperties keys: {list(sample_activity.get('typeProperties', {}).keys())}")

# Try to find the notebook reference in different possible locations
type_props = sample_activity.get('typeProperties', {})

# TridentNotebook uses notebookId (GUID)
if 'notebookId' in type_props:
    sample_ref = type_props['notebookId']
    sample_workspace = type_props.get('workspaceId', workspace_id)
    print(f"\n✓ Found TridentNotebook structure:")
    print(f"  notebookId: {sample_ref}")
    print(f"  workspaceId: {sample_workspace}")
elif 'notebook' in type_props:
    sample_ref = type_props['notebook']['referenceName']
elif 'notebookPath' in type_props:
    sample_ref = type_props['notebookPath']
elif 'itemId' in type_props:
    sample_ref = type_props['itemId']
elif 'itemName' in type_props:
    sample_ref = type_props['itemName']
else:
    print(f"\n⚠️  Full typeProperties structure:")
    print(json.dumps(type_props, indent=2))
    raise RuntimeError("Cannot find notebook reference in activity. See structure above.")

# Store the format for later
DISCOVERED_FORMAT = sample_ref
DISCOVERED_STRUCTURE = 'trident' if 'notebookId' in type_props else 'synapse'

print(f"\n✓ Discovered notebook reference format:")
print(f"  Activity name: {sample_activity['name']}")
print(f"  Reference value: '{sample_ref}'")
print(f"  Reference type: {type(sample_ref).__name__}")
print(f"  Length: {len(sample_ref)} characters")



if len(sample_ref) == 36 and '-' in sample_ref:print(f"\n📋 This is the format Fabric expects for notebook references")

    print(f"  Format appears to be: GUID")

else:    print(f"  Format appears to be: Plain notebook name")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 4: LOAD OUR PIPELINE JSON AND MAP NOTEBOOKS
# ════════════════════════════════════════════════════════════════════════

PIPELINE_JSON_PATH = "/lakehouse/default/Files/CrossIndustry_Pipeline.json"

try:
    with open(PIPELINE_JSON_PATH, 'r') as f:
        our_pipeline = json.load(f)
    print(f"✓ Loaded CrossIndustry_Pipeline.json")
except FileNotFoundError:
    print(f"✗ Upload CrossIndustry_Pipeline.json to Lakehouse Files/ first")
    print(f"\n  1. Attach a Lakehouse to this notebook")
    print(f"  2. Upload the JSON file to Files/ folder")
    print(f"  3. Re-run this cell")
    raise

# Build notebook name → ID mapping
items_df = fabric.list_items()
notebooks = items_df[items_df["Type"] == "Notebook"]

print(f"\nMapping {len(our_pipeline['properties']['activities'])} activities to notebook references...")
print(f"  Total notebooks in workspace: {len(notebooks)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 5: APPLY DISCOVERED FORMAT TO PIPELINE (IF NEEDED)
# ════════════════════════════════════════════════════════════════════════

print(f"\n🔍 Converting pipeline to Fabric format...\n")

# Build name → GUID mapping
notebook_guid_map = {}
for _, row in notebooks.iterrows():
    notebook_guid_map[row["Display Name"]] = str(row["Id"])

print(f"Discovered structure: {DISCOVERED_STRUCTURE}")
print(f"Converting activities to TridentNotebook format...\n")

# Convert all activities to Fabric TridentNotebook format
for activity in our_pipeline['properties']['activities']:
    # Get the notebook name from the old format
    nb_name = activity['typeProperties']['notebook']['referenceName']
    nb_guid = notebook_guid_map.get(nb_name)
    
    if not nb_guid:
        print(f"  ✗ {activity['name']}: Notebook '{nb_name}' not found!")
        raise RuntimeError(f"Notebook '{nb_name}' doesn't exist in workspace")
    
    # Convert to TridentNotebook format
    activity['type'] = 'TridentNotebook'
    activity['typeProperties'] = {
        'notebookId': nb_guid,
        'workspaceId': workspace_id
    }
    
    print(f"  ✓ {activity['name']}: {nb_name} → {nb_guid[:8]}...")

print(f"\n✓ All {len(our_pipeline['properties']['activities'])} activities converted to Fabric format")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 6: CREATE PIPELINE VIA REST API
# ════════════════════════════════════════════════════════════════════════

PIPELINE_NAME = our_pipeline.get('name', 'CrossIndustry_Deployment_Pipeline')

# Check if already exists
items_df = fabric.list_items()
existing = items_df[
    (items_df["Type"] == "DataPipeline") & 
    (items_df["Display Name"] == PIPELINE_NAME)
]

if not existing.empty:
    print(f"⚠️  Pipeline '{PIPELINE_NAME}' already exists")
    print(f"   Delete it manually or rename in JSON, then re-run")
    raise RuntimeError("Pipeline name conflict")

# Encode pipeline as base64
pipeline_b64 = base64.b64encode(json.dumps(our_pipeline).encode()).decode()

payload = {
    "displayName": PIPELINE_NAME,
    "description": our_pipeline.get('properties', {}).get('description', 'Cross-Industry pipeline'),
    "type": "DataPipeline",
    "definition": {
        "parts": [{
            "path": "pipeline-content.json",
            "payload": pipeline_b64,
            "payloadType": "InlineBase64"
        }]
    }
}

url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

print(f"\nCreating pipeline '{PIPELINE_NAME}'...")
resp = requests.post(url, headers=headers, json=payload, timeout=60)

if resp.status_code in (200, 201, 202):
    result = resp.json()
    print(f"\n✅ Pipeline created successfully!")
    print(f"   ID: {result.get('id', 'N/A')}")
    print(f"\n🚀 Next: Go to Data Factory → Find '{PIPELINE_NAME}' → Click Run")
else:
    print(f"\n✗ Failed: {resp.status_code}")
    print(f"   {resp.text[:1000]}")
    raise RuntimeError("Pipeline creation failed")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 7: VERIFY DEPLOYMENT
# ════════════════════════════════════════════════════════════════════════

print("\nVerifying deployment...")
time.sleep(5)

items_df = fabric.list_items()
pipelines = items_df[items_df["Type"] == "DataPipeline"]
deployed = pipelines[pipelines["Display Name"] == PIPELINE_NAME]

if not deployed.empty:
    info = deployed.iloc[0]
    print(f"\n✓ Verified:")
    print(f"  Name: {info['Display Name']}")
    print(f"  ID: {info['Id']}")
    
    print(f"\n  Activities:")
    for act in our_pipeline['properties']['activities']:
        deps = act.get('dependsOn', [])
        dep_str = f" (after: {', '.join(d['activity'] for d in deps)})" if deps else ""
        print(f"    • {act['name']}{dep_str}")
    
    print(f"\n{'═'*70}")
    print(f"DEPLOYMENT COMPLETE")
    print(f"{'═'*70}")
    print(f"\n🚀 Go to: Fabric → Data Factory → {PIPELINE_NAME} → Run")
    print(f"\n{'═'*70}")
else:
    print(f"\n⚠ Not visible yet (may still be provisioning). Check in 1 minute.")